# Caching API responses
Use this notebook to populate `RESPONSE_CACHE_FILE` with responses from that API that can later be used in the training loop

In [2]:
import base64
import io
import json
import os
from hashlib import md5
from typing import Dict, Optional

import requests
from PIL import Image


In [3]:
TARGET_SERVER_URL = "https://wstud-mmml26-classification.web.webis.de"
RESPONSE_CACHE_FILE = "../cache/responses.json"

In [4]:
session = requests.Session()


### Parsing ###

def parse_img(img: Image.Image) -> tuple[str, str]:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    img_base64_bytes = base64.b64encode(buf.getvalue())
    img_base64 = img_base64_bytes.decode("utf-8")
    img_hash = md5(img_base64_bytes).hexdigest()
    return img_base64, img_hash


### Cache ###

def get_cache(img_hash) -> Optional[Dict]:
    with open(RESPONSE_CACHE_FILE, "r") as f:
        cache_content = json.load(f)
        return cache_content.get(img_hash, None)


def set_cache(img_hash, api_response) -> None:
    with open(RESPONSE_CACHE_FILE, "r") as f:
        cache_content = json.load(f)
        cache_content[img_hash] = api_response
    with open(RESPONSE_CACHE_FILE, "w") as f:
        json.dump(cache_content, f)


### Network ###

def query_api(img_base64) -> Dict:
    try:
        resp = session.post(
            TARGET_SERVER_URL + "/predict",
            json={"image": img_base64, "classes": ["class1", "class2", "class3", "class4"]},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"Request failed {e}")
        return {}


### User call ###

def get_score(img) -> Dict:
    img_base64, img_hash = parse_img(img)
    cache_hit = get_cache(img_hash)
    if cache_hit:
        return cache_hit
    api_response = query_api(img_base64)
    set_cache(img_hash, api_response)
    return api_response

In [5]:
dir_path = "../img"
images = [Image.open(f"{dir_path}/{p}") for p in os.listdir(dir_path)]

In [6]:
for i in images:
    score = get_score(i)
    print(score)

{'classes': {'class1': 0.3910961449146271, 'class2': 0.6089038848876953, 'class3': 0.9323794841766357, 'class4': 0.06762055307626724}}
{'classes': {'class1': 0.11249769479036331, 'class2': 0.8875023126602173, 'class3': 0.9596433043479919, 'class4': 0.04035666584968567}}
{'classes': {'class1': 0.1592283844947815, 'class2': 0.8407716155052185, 'class3': 0.3030906319618225, 'class4': 0.6969093680381775}}
{'classes': {'class1': 0.8734389543533325, 'class2': 0.12656109035015106, 'class3': 0.5671042799949646, 'class4': 0.4328957200050354}}
{'classes': {'class1': 0.9718536138534546, 'class2': 0.02814633585512638, 'class3': 0.38879862427711487, 'class4': 0.6112013459205627}}
